# Exercises XP Gold: Guided Student Notebook

This guided notebook follows the **exact exercise on the platform**. Cells marked **PREFILLED** are for execution only. Cells marked **To-Do** require your action. When a written answer is required, the **To-Do** appears inside a markdown cell. When code is required, the **To-Do** appears inside a code cell as comments.

Learning points appear only for key concepts to support intuition and transfer to other AI topics.



## Reference from the exercise

**What you will learn**  
- Implement advanced techniques to improve model performance.
- Understand the impact of hyperparameter tuning on model accuracy.
- Explore ensemble methods.
- Apply transfer learning to leverage pre-trained models.
- Use data augmentation to enhance dataset diversity.

**What you will create**  
A tuned neural network with improved accuracy. An ensemble model. A transfer learning pipeline. A data augmentation strategy. A comparison of model performance before and after advanced techniques.


## Exercise 1: Hyperparameter Tuning for Neural Networks

**As stated in the exercise**  
Objective: Improve model performance by tuning hyperparameters.  
Instructions: Choose a neural architecture such as a simple MLP or CNN. Identify hyperparameters such as learning rate, batch size, number of layers, and number of neurons. Use grid search or random search to explore combinations. Train with the best hyperparameters and compare to a baseline. Write a short summary about the improvement.


**Guidance**  
Use MNIST for quick iterations. Start from a baseline MLP. Tune a small search space to keep runtime reasonable. Track validation accuracy for model selection.

**Learning point**  
Search quality depends on search space and budget. Random search is often more efficient than grid when many hyperparameters matter.


In [1]:
# PREFILLED: just execute
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

np.random.seed(7)
tf.random.set_seed(7)

print("TensorFlow:", tf.__version__)
(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train = x_train.astype("float32")/255.0
x_test  = x_test.astype("float32")/255.0
y_train_oh = to_categorical(y_train, 10)
y_test_oh  = to_categorical(y_test, 10)
x_train.shape, y_train_oh.shape, x_test.shape, y_test_oh.shape

TensorFlow: 2.20.0
11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


((60000, 28, 28), (60000, 10), (10000, 28, 28), (10000, 10))

In [2]:
# PREFILLED: just execute
def build_baseline_mlp(hidden1=128, hidden2=64, dropout=0.0, lr=1e-3):
    model = models.Sequential([
        layers.Input(shape=(28,28)),
        layers.Flatten(),
        layers.Dense(hidden1, activation="relu"),
        layers.Dropout(dropout),
        layers.Dense(hidden2, activation="relu"),
        layers.Dense(10, activation="softmax"),
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(lr),
                  loss="categorical_crossentropy",
                  metrics=["accuracy"])
    return model

baseline = build_baseline_mlp()
history_base = baseline.fit(x_train, y_train_oh, validation_split=0.1,
                            epochs=3, batch_size=128, verbose=2)
base_test = baseline.evaluate(x_test, y_test_oh, verbose=0)
print({"baseline_test_acc": float(base_test[1])})

Epoch 1/3
422/422 - 4s - 9ms/step - accuracy: 0.8995 - loss: 0.3533 - val_accuracy: 0.9617 - val_loss: 0.1428
Epoch 2/3
422/422 - 2s - 4ms/step - accuracy: 0.9589 - loss: 0.1394 - val_accuracy: 0.9702 - val_loss: 0.1047
Epoch 3/3
422/422 - 2s - 4ms/step - accuracy: 0.9712 - loss: 0.0955 - val_accuracy: 0.9737 - val_loss: 0.0919
{'baseline_test_acc': 0.9711999893188477}


In [ ]:
# To-Do: define a small search space and implement random or grid search
# Example search space suggestions:
# hidden1_choices = [128, 256]
# hidden2_choices = [64, 128]
# dropout_choices = [0.0, 0.2, 0.3]
# lr_choices = [1e-3, 5e-4]
# batch_choices = [64, 128]
#
# To-Do: iterate over combinations or sample randomly, train for few epochs (e.g., 2)
# keep track of best validation accuracy and corresponding config
# best_cfg = {...}; best_val_acc = ...
# print("best config:", best_cfg, "best val acc:", best_val_acc)

In [3]:
from sklearn.model_selection import ParameterGrid
import random

# Define a small search space
hidden1_choices = [128, 256]
hidden2_choices = [64, 128]
dropout_choices = [0.0, 0.2, 0.3]
lr_choices = [1e-3, 5e-4]
batch_choices = [64, 128]

param_grid = {
    'hidden1': hidden1_choices,
    'hidden2': hidden2_choices,
    'dropout': dropout_choices,
    'lr': lr_choices,
    'batch_size': batch_choices
}

# Create a list of all possible combinations (grid search equivalent for small spaces)
# Or, for random search, sample from this grid
all_combinations = list(ParameterGrid(param_grid))

# For demonstration, let's randomly sample a few combinations
# In a real scenario, you might run more or use a dedicated random search library
num_samples = 10 # Number of random samples to try
random_configs = random.sample(all_combinations, min(num_samples, len(all_combinations)))

best_val_acc = -1
best_cfg = None

print(f"Testing {len(random_configs)} random configurations...")

for i, config in enumerate(random_configs):
    print(f"\n--- Training config {i+1}/{len(random_configs)}: {config} ---")

    # Extract hyperparameters for model building and training
    current_hidden1 = config['hidden1']
    current_hidden2 = config['hidden2']
    current_dropout = config['dropout']
    current_lr = config['lr']
    current_batch_size = config['batch_size']

    # Build the model
    model = build_baseline_mlp(hidden1=current_hidden1, hidden2=current_hidden2,
                               dropout=current_dropout, lr=current_lr)

    # Train for a few epochs
    history = model.fit(x_train, y_train_oh, validation_split=0.1,
                        epochs=2, batch_size=current_batch_size, verbose=0) # Set verbose to 0 to suppress output

    # Get the validation accuracy from the last epoch
    current_val_acc = history.history['val_accuracy'][-1]
    print(f"Validation accuracy: {current_val_acc:.4f}")

    if current_val_acc > best_val_acc:
        best_val_acc = current_val_acc
        best_cfg = config
print("Best config found:", best_cfg)
print("Best validation accuracy:", f"{best_val_acc:.4f}")

Testing 10 random configurations...

--- Training config 1/10: {'batch_size': 64, 'dropout': 0.2, 'hidden1': 256, 'hidden2': 128, 'lr': 0.0005} ---
Validation accuracy: 0.9732

--- Training config 2/10: {'batch_size': 64, 'dropout': 0.0, 'hidden1': 256, 'hidden2': 64, 'lr': 0.001} ---
Validation accuracy: 0.9770

--- Training config 3/10: {'batch_size': 128, 'dropout': 0.2, 'hidden1': 128, 'hidden2': 128, 'lr': 0.001} ---
Validation accuracy: 0.9733

--- Training config 4/10: {'batch_size': 128, 'dropout': 0.0, 'hidden1': 256, 'hidden2': 64, 'lr': 0.0005} ---
Validation accuracy: 0.9690

--- Training config 5/10: {'batch_size': 128, 'dropout': 0.3, 'hidden1': 256, 'hidden2': 64, 'lr': 0.0005} ---
Validation accuracy: 0.9695

--- Training config 6/10: {'batch_size': 64, 'dropout': 0.3, 'hidden1': 128, 'hidden2': 128, 'lr': 0.001} ---
Validation accuracy: 0.9747

--- Training config 7/10: {'batch_size': 64, 'dropout': 0.2, 'hidden1': 256, 'hidden2': 64, 'lr': 0.001} ---
Validation accura

In [5]:
# To-Do: retrain a final model with best_cfg for more epochs and compare to baseline on test set
best_cfg_without_batch_lr = {k: v for k, v in best_cfg.items() if k != 'batch_size'}
final_model = build_baseline_mlp(**best_cfg_without_batch_lr)  # map your keys
history_final = final_model.fit(x_train, y_train_oh, validation_split=0.1,
                                epochs=10, batch_size=best_cfg['batch_size'], verbose=2)
test_loss, test_acc = final_model.evaluate(x_test, y_test_oh, verbose=0)
print({"baseline_test_acc": float(base_test[1]), "tuned_test_acc": float(test_acc)})

Epoch 1/10
844/844 - 7s - 8ms/step - accuracy: 0.9248 - loss: 0.2592 - val_accuracy: 0.9668 - val_loss: 0.1090
Epoch 2/10
844/844 - 7s - 8ms/step - accuracy: 0.9691 - loss: 0.1016 - val_accuracy: 0.9732 - val_loss: 0.0892
Epoch 3/10
844/844 - 5s - 6ms/step - accuracy: 0.9800 - loss: 0.0660 - val_accuracy: 0.9762 - val_loss: 0.0826
Epoch 4/10
844/844 - 7s - 8ms/step - accuracy: 0.9866 - loss: 0.0451 - val_accuracy: 0.9775 - val_loss: 0.0866
Epoch 5/10
844/844 - 10s - 12ms/step - accuracy: 0.9902 - loss: 0.0320 - val_accuracy: 0.9763 - val_loss: 0.0908
Epoch 6/10
844/844 - 5s - 6ms/step - accuracy: 0.9913 - loss: 0.0270 - val_accuracy: 0.9763 - val_loss: 0.1035
Epoch 7/10
844/844 - 6s - 8ms/step - accuracy: 0.9929 - loss: 0.0219 - val_accuracy: 0.9783 - val_loss: 0.1014
Epoch 8/10
844/844 - 6s - 7ms/step - accuracy: 0.9938 - loss: 0.0182 - val_accuracy: 0.9740 - val_loss: 0.1209
Epoch 9/10
844/844 - 6s - 8ms/step - accuracy: 0.9947 - loss: 0.0156 - val_accuracy: 0.9753 - val_loss: 0.1145

**To-Do:** Write 3–5 sentences explaining how hyperparameter tuning changed the learning dynamics and accuracy compared to the baseline.


L'optimisation des hyperparamètres a permis une légère amélioration de la précision sur les données de test, passant d'environ 97,12 % pour le modèle de base à 97,74 % pour le modèle optimisé. La meilleure configuration, comprenant une première couche cachée plus large, une taille de lot plus petite et l'absence de dropout, a permis au modèle d'atteindre de meilleures performances. Bien que le modèle optimisé ait été entraîné pendant un plus grand nombre d'époques (10 contre 3), sa précision de validation a montré une amélioration continue sur une période plus longue, indiquant un processus d'apprentissage plus efficace avant de présenter de légers signes de surapprentissage potentiel vers la fin des époques. Cela suggère que les hyperparamètres optimisés ont permis d'obtenir un modèle plus robuste et légèrement plus précis.

## Exercise 2: Building an Ensemble Model

**As stated in the exercise**  
Objective: Combine predictions from multiple models to improve accuracy.  

Instructions: Train at least three different models such as a decision tree, a random forest, and a neural network on the same dataset. Use voting to combine predictions. Evaluate the ensemble and compare to individual models. Try weighted voting or stacking. Write a short reflection on why ensembles often outperform individuals.


**Guidance**  
Use the sklearn digits dataset for speed. Train DecisionTree, RandomForest, and MLPClassifier. Start with hard voting and then try weights.


In [6]:
# PREFILLED: just execute
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

digits = load_digits()
X = digits.data.astype("float32")/16.0
y = digits.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler(with_mean=True, with_std=True)
X_trs = scaler.fit_transform(X_tr)
X_tes = scaler.transform(X_te)

print("Digits:", X_trs.shape, X_tes.shape)

Digits: (1437, 64) (360, 64)


In [9]:
# To-Do: train three base models
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
#
clf_tree = DecisionTreeClassifier(random_state=42)
clf_rf = RandomForestClassifier(random_state=42)
clf_mlp = MLPClassifier(random_state=42)
#
# Fit the models
clf_tree.fit(X_trs, y_tr)
clf_rf.fit(X_trs, y_tr)
clf_mlp.fit(X_trs, y_tr)

preds_tree = clf_tree.predict(X_tes)
preds_rf   = clf_rf.predict(X_tes)
preds_mlp  = clf_mlp.predict(X_tes)
print({"acc_tree": round(accuracy_score(y_te, preds_tree),2),
       "acc_rf": round(accuracy_score(y_te, preds_rf),2),
       "acc_mlp": round(accuracy_score(y_te, preds_mlp),2)})

{'acc_tree': 0.82, 'acc_rf': 0.96, 'acc_mlp': 0.97}


In [10]:
# To-Do: build a VotingClassifier and evaluate
from sklearn.ensemble import VotingClassifier
ensemble = VotingClassifier(estimators=[("tree", clf_tree), ("rf", clf_rf), ("mlp", clf_mlp)], voting="hard")
ensemble.fit(X_trs, y_tr)
preds_ens = ensemble.predict(X_tes)
print({"acc_ensemble": round(accuracy_score(y_te, preds_ens),2)})

{'acc_ensemble': 0.96}


**To-Do:** Try weighted voting or stacking. Report the best ensemble accuracy and compare it to the best single model.


**Learning point**  
Ensembles reduce variance by averaging diverse learners. Diversity matters. Weighted voting or stacking can exploit stronger models more.


## Exercise 3: Transfer Learning with Pre-Trained Models

**As stated in the exercise**  
Objective: Leverage a pre-trained model to solve a new problem.  

Instructions: Choose a pre-trained model such as ResNet, VGG, or MobileNet. Replace the final layer to adapt to your classes. Freeze initial layers and train only the new layers. Fine tune by unfreezing some earlier layers with a lower learning rate. Write an explanation of how transfer learning saves time and compute.


**Guidance**  
Use CIFAR-10 for a quick demo and MobileNetV2 as a pre-trained feature extractor. Resize images to 96x96. Start with the base frozen, then fine tune the top blocks.

**Learning point**  
Transfer learning reuses general visual features learned on large datasets. Freezing stabilizes early training. Fine tuning aligns features to the new domain.


In [11]:
# PREFILLED: just execute
import tensorflow as tf
from tensorflow.keras import layers, models

from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical

(x_tr, y_tr), (x_te, y_te) = cifar10.load_data()
x_tr = x_tr.astype("float32")/255.0
x_te = x_te.astype("float32")/255.0

# Resize to 96x96 for MobileNetV2
x_tr = tf.image.resize(x_tr, (96,96)).numpy()
x_te = tf.image.resize(x_te, (96,96)).numpy()
y_tr_oh = to_categorical(y_tr, 10)
y_te_oh = to_categorical(y_te, 10)

print("CIFAR10:", x_tr.shape, y_tr_oh.shape)

In [ ]:
# To-Do: build a transfer model
# base = tf.keras.applications.MobileNetV2(include_top=False, input_shape=(96,96,3), weights="imagenet")
# base.trainable = False  # freeze
# inputs = layers.Input(shape=(96,96,3))
# x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
# x = base(x, training=False)
# x = layers.GlobalAveragePooling2D()(x)
# x = layers.Dense(128, activation="relu")(x)
# outputs = layers.Dense(10, activation="softmax")(x)
# model_tl = ...
# hist1 = model_tl.fit(x_tr, y_tr_oh, validation_split=0.1, epochs=3, batch_size=128, verbose=2)

In [ ]:
# To-Do: fine tune - unfreeze some top layers and train with a lower LR
# base.trainable = True
# for layer in base.layers[:-20]:
#     layer.trainable = False
# ...
# test_loss, test_acc = model_tl.evaluate(x_te, y_te_oh, verbose=0)
# print({"transfer_test_acc": float(test_acc)})

**To-Do:** Write 3–5 sentences explaining why transfer learning reduces training time and data requirements.


![image.png](https://github.com/user-attachments/assets/5786486a-7c4f-442d-8ca6-c61861d77f22)

## Exercise 4: Data Augmentation for Image Datasets

**As stated in the exercise**  
Objective: Increase dataset diversity and improve generalization.  

Instructions: Choose an image dataset such as CIFAR-10. Apply augmentation such as rotation, flipping, cropping, and color jitter. Train a model on augmented data and compare to a model trained on the original data. Experiment with combinations. Write a short summary on how augmentation helps prevent overfitting.


**Guidance**  
Use Keras preprocessing layers to compose augmentation. Keep augmentations moderate to avoid distribution shift.

**Learning point**  
Augmentation encodes invariances and expands the effective training set. Excessive augmentation can harm performance.


In [12]:
# PREFILLED: just execute
from tensorflow.keras import layers

augment = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])

In [14]:
# To-Do: visualize a few augmented samples
# import matplotlib.pyplot as plt
# idx = np.random.choice(...)
# for i, k in enumerate(idx):
#     img = x_tr[k:k+1]
#     aug = ...
#     plt.figure(figsize=(2.5,2.5)); plt.imshow(aug); plt.axis("off"); plt.show()

NameError: name 'x_tr' is not defined

In [ ]:
# To-Do: build 2 models - baseline (no augmentation) and augmented (apply augment in the model)
# def build_cnn_with_aug(use_aug=False):
#     inputs = layers.Input(shape=(96,96,3))
#     x = ...
#       ...
#     outputs = layers.Dense(10, activation="softmax")(x)
#     model = models.Model(inputs, outputs)
#     model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
#     return model
#
# m_base = build_cnn_with_aug(use_aug=False)
# h_base = m_base.fit(x_tr, y_tr_oh, validation_split=0.1, epochs=3, batch_size=128, verbose=2)
# base_acc = float(m_base.evaluate(x_te, y_te_oh, verbose=0)[1])
#
# m_aug = build_cnn_with_aug(use_aug=True)
# h_aug = m_aug.fit(x_tr, y_tr_oh, validation_split=0.1, epochs=3, batch_size=128, verbose=2)
# aug_acc = float(m_aug.evaluate(x_te, y_te_oh, verbose=0)[1])
# print({"base_acc": base_acc, "aug_acc": aug_acc})

**To-Do:** Write 3–5 sentences summarizing how augmentation changed generalization and when to adjust augmentation strength.


## Exercise 5: Model Performance Comparison with Advanced Techniques

**As stated in the exercise**  
Objective: Compare performance before and after applying advanced techniques.  

Instructions: Train a baseline model without advanced techniques. Apply at least two techniques such as tuning and augmentation. Train the improved model and evaluate on the same test set. Create a table or chart comparing accuracy and loss. Write a short reflection on the effectiveness.


**Guidance**  
Collect results consistently: same test split, identical metrics, and equal evaluation protocol.

**Learning point**  
Keep a fixed evaluation harness. Changing data or metrics confounds comparisons.


In [ ]:
# Pre-filled: assembling a comparison table from your earlier results
# results = [
#     {"model": "baseline_mlp_mnist", "test_acc": float(base_test[1])},
#     {"model": "tuned_mlp_mnist", "test_acc": float(test_acc)},  # from Exercise 1
#     {"model": "ensemble_digits", "test_acc": ...},              # from Exercise 2
#     {"model": "transfer_cifar10", "test_acc": ...},             # from Exercise 3
#     {"model": "augmented_cifar10", "test_acc": ...},            # from Exercise 4
# ]
# # Simple bar chart
# labels = [r["model"] for r in results]
# vals = [r["test_acc"] for r in results]
# plt.figure(figsize=(8,4))
# plt.bar(range(len(vals)), vals)
# plt.xticks(range(len(vals)), labels, rotation=20)
# plt.ylabel("test accuracy")
# plt.title("Performance comparison")
# plt.tight_layout()
# plt.show()

**To-Do:** Write 3–5 sentences reflecting on which techniques delivered the largest gains and why.
